# 01 新聞資料品質檢查

本 notebook 僅讀取本地 SQLite 資料庫，不呼叫 Groq 或任何外部 API。

檢查項目：
1. 原始新聞基本統計（筆數、來源、日期範圍）
2. 每日爬取量趨勢
3. 重複新聞檢查（相同 URL）
4. 標題文字長度分布
5. LLM 分析覆蓋率與失敗原因
6. 標的分布（target coverage）

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "backend" / "data" / "twstock_sentiment.db"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.unicode_minus"] = False

print(f"DB: {DB_PATH}  (exists={DB_PATH.exists()})")

## 1. 載入資料

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    raw = pd.read_sql_query(
        "SELECT id, title, url, published_at, source FROM raw_news ORDER BY published_at",
        conn,
    )
    llm = pd.read_sql_query(
        "SELECT news_id, status, target, news_type, sentiment, confidence, model_name FROM llm_news_analysis",
        conn,
    )

raw["published_at"] = pd.to_datetime(raw["published_at"], utc=True, errors="coerce")
raw["date"] = raw["published_at"].dt.date
raw["title_len"] = raw["title"].str.len()

print(f"raw_news rows   : {len(raw):,}")
print(f"llm_news rows   : {len(llm):,}")
print(f"date range      : {raw['published_at'].min().date()} ~ {raw['published_at'].max().date()}")
raw.head()

## 2. 來源分布

In [ ]:
source_dist = raw["source"].value_counts().rename_axis("source").reset_index(name="count")
source_dist["ratio"] = source_dist["count"] / source_dist["count"].sum()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=source_dist, x="source", y="count", ax=ax, hue="source", legend=False, palette="Set2")
ax.set_title("新聞來源分布")
ax.set_xlabel("來源")
ax.set_ylabel("筆數")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "news_source_distribution.png", dpi=160)
plt.show()
display(source_dist)

## 3. 每日爬取量趨勢

In [ ]:
daily_count = raw.groupby("date").size().reset_index(name="count")
daily_count["date"] = pd.to_datetime(daily_count["date"])

fig, ax = plt.subplots()
ax.bar(daily_count["date"], daily_count["count"], width=0.8, color="steelblue", alpha=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
plt.xticks(rotation=45)
ax.set_title("每日新聞爬取量")
ax.set_xlabel("日期")
ax.set_ylabel("筆數")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "news_daily_count.png", dpi=160)
plt.show()
print(f"平均每日: {daily_count['count'].mean():.1f} 則  |  最多: {daily_count['count'].max()} 則  |  共 {len(daily_count)} 個日期")

## 4. 重複新聞檢查

In [ ]:
dup_url = raw[raw.duplicated(subset=["url"], keep=False)].sort_values("url")
dup_title = raw[raw.duplicated(subset=["title"], keep=False)].sort_values("title")

print(f"重複 URL 數量   : {len(dup_url):,} 筆（{dup_url['url'].nunique()} 個不重複 URL 出現 2 次以上）")
print(f"重複標題數量    : {len(dup_title):,} 筆（{dup_title['title'].nunique()} 個不重複標題出現 2 次以上）")

if len(dup_url) > 0:
    display(Markdown("### 重複 URL 範例（前 5 個）"))
    display(dup_url[["id","title","url","published_at"]].head(10))
else:
    display(Markdown("✅ 無重複 URL — 爬蟲的 URL 去重機制正常運作"))

## 5. 標題文字長度分布

In [ ]:
fig, ax = plt.subplots()
ax.hist(raw["title_len"].dropna(), bins=40, color="teal", edgecolor="white", alpha=0.85)
ax.axvline(raw["title_len"].median(), color="red", linestyle="--", label=f"中位數 {raw['title_len'].median():.0f} 字")
ax.set_title("新聞標題長度分布")
ax.set_xlabel("字元數")
ax.set_ylabel("篇數")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "news_title_length.png", dpi=160)
plt.show()

print(raw["title_len"].describe().to_string())

## 6. LLM 分析覆蓋率

In [ ]:
analyzed_ids = set(llm["news_id"])
raw["llm_status"] = raw["id"].map(
    llm.set_index("news_id")["status"]
).fillna("pending")

status_counts = raw["llm_status"].value_counts().rename_axis("status").reset_index(name="count")
status_counts["ratio"] = status_counts["count"] / status_counts["count"].sum()

fig, ax = plt.subplots(figsize=(7, 4))
colors = {"success": "#4a7c59", "failed": "#b7472a", "pending": "#708090"}
bar_colors = [colors.get(s, "gray") for s in status_counts["status"]]
sns.barplot(data=status_counts, x="status", y="count", ax=ax, palette=bar_colors, hue="status", legend=False)
ax.set_title("LLM 分析狀態分布")
ax.set_xlabel("狀態")
ax.set_ylabel("筆數")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "llm_status_distribution.png", dpi=160)
plt.show()

display(status_counts)
success_n = status_counts.loc[status_counts["status"]=="success", "count"].sum()
print(f"LLM 成功率：{success_n / len(raw):.1%}")

## 7. 標的分布（Target Coverage）

In [ ]:
success_llm = llm[llm["status"] == "success"].copy()
target_dist = (
    success_llm["target"]
    .value_counts()
    .rename_axis("target")
    .reset_index(name="count")
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=target_dist, x="target", y="count", ax=ax, hue="target", legend=False, palette="Blues_r")
ax.set_title("前 20 大標的新聞筆數（LLM 成功分析）")
ax.set_xlabel("標的")
ax.set_ylabel("筆數")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "target_distribution.png", dpi=160)
plt.show()

display(target_dist)

## 8. 資料品質摘要

In [ ]:
summary = {
    "原始新聞總筆數": len(raw),
    "日期範圍": f"{raw['published_at'].min().date()} ~ {raw['published_at'].max().date()}",
    "資料天數": raw["date"].nunique(),
    "來源種類": raw["source"].nunique(),
    "重複 URL 筆數": len(dup_url),
    "LLM 成功分析": int(status_counts.loc[status_counts["status"]=="success", "count"].sum() if "success" in status_counts["status"].values else 0),
    "LLM 失敗": int(status_counts.loc[status_counts["status"]=="failed", "count"].sum() if "failed" in status_counts["status"].values else 0),
    "待分析 (pending)": int(status_counts.loc[status_counts["status"]=="pending", "count"].sum() if "pending" in status_counts["status"].values else 0),
    "平均標題長度 (字)": f"{raw['title_len'].mean():.1f}",
    "出現的股票代號": success_llm["target"].nunique(),
}

summary_df = pd.DataFrame(list(summary.items()), columns=["項目", "數值"])
summary_df.to_csv(TABLE_DIR / "news_data_quality_summary.csv", index=False, encoding="utf-8-sig")
display(summary_df)